# Part-Conditioned I-JEPA (TI-JEPA Style)
## A Joint-Embedding Predictive Architecture Guided by Object-Part Text Prompts

### Architecture Overview
This notebook implements a **Part-Conditioned I-JEPA** that fuses visual and text semantics
to perform **object-centric part representation**. It follows the TI-JEPA paradigm shown in the
reference figure: the context encoder output passes through a **t2i cross-attention layer**
where image patch tokens (Queries) attend to CLIP text features (Keys/Values).

**Golden Rule:** We never rewrite ViT blocks or patch embedders from scratch.
All backbone components are imported directly from `src/models/vision_transformer.py`
in this repository.

### What This Notebook Covers
| Task | Description |
|------|-------------|
| 1    | PartImageNet COCO-JSON Data Loader |
| 2    | Text-Conditioned Context Encoder (t2i Cross-Attention) |
| 3    | Semantic Target Masking from Bounding Boxes |
| 4    | Full Forward Pass + Loss Computation |
| 5    | Cross-Attention Map Visualization (Zero-Shot Part Localization) |

---
## Task 1 — Setup: Imports & Environment

We import the **local I-JEPA backbone components** directly:
- `vit_small`, `vit_base` — factory functions that build `VisionTransformer` (encoder)
- `vit_predictor` — builds `VisionTransformerPredictor`
- `apply_masks` — utility that gathers patch tokens by index
- `MaskCollator` — the original random block masking collator (we will override its predicate mask logic in Task 3)

HuggingFace `transformers` is used **only** for the CLIP text encoder.
No ViT block is written by hand.

In [ ]:
# ── Kaggle / Colab Environment Setup ───────────────────────────────────
!pip install -q transformers huggingface_hub

In [ ]:
# ── Standard Library ────────────────────────────────────────────────────────
import os
import sys
import json
import copy
import math
from pathlib import Path
from collections import defaultdict

# ── Add repo root to path so local src/ imports resolve ─────────────────────
REPO_ROOT = Path(".").resolve()          # assumes notebook lives in repo root
sys.path.insert(0, str(REPO_ROOT))

# ── PyTorch Core ────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Torchvision ──────────────────────────────────────────────────────────────
import torchvision.transforms as T
from PIL import Image

# ── NumPy / Matplotlib ───────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LOCAL I-JEPA IMPORTS  ← THE GOLDEN RULE: always import, never rewrite
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from src.models.vision_transformer import (
    VisionTransformer,            # full ViT encoder class
    VisionTransformerPredictor,   # predictor class (processes mask tokens)
    vit_small,                    # factory: embed_dim=384, depth=12, heads=6
    vit_base,                     # factory: embed_dim=768, depth=12, heads=12
    vit_predictor,                # factory for predictor head
)
from src.masks.multiblock import MaskCollator   # original random block masker
from src.masks.utils import apply_masks          # gather patches by index list

# ── HuggingFace CLIP Text Encoder ────────────────────────────────────────────
# We use ONLY the text side of CLIP (CLIPTextModel + CLIPTokenizer).
# The visual backbone comes from I-JEPA's VisionTransformer above.
from transformers import CLIPTextModel, CLIPTokenizer

# ── Device Setup ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

---
## Task 1 — PartImageNet Dataset (COCO-Style JSON Parser)

### PartImageNet Annotation Structure
PartImageNet ships COCO-format JSON files. Each annotation entry contains:
```json
{
  "id": 12345,
  "image_id": 789,
  "category_id": 3,
  "bbox": [x, y, width, height],   ← COCO format: top-left corner
  "segmentation": [[...]],
  "area": 4096
}
```
The `categories` list maps `category_id` → `{"id": 3, "name": "Quadruped head"}`.

### Why the Bounding Box Controls Target Masking
In standard I-JEPA the **predictor target** is a *randomly* sampled rectangular block of
patches. In our Part-Conditioned variant we **replace** that random block with the set of
patch indices that fall *inside the semantic part bounding box*.

For a 224×224 image with patch_size=16, the patch grid is **14×14 = 196 patches**.  
Given a bbox `[x, y, w, h]` in pixel space:
```
patch_col_start = x  // patch_size
patch_row_start = y  // patch_size
patch_col_end   = (x + w) // patch_size
patch_row_end   = (y + h) // patch_size
```
Every patch index `(row * grid_w + col)` in that range becomes the **semantic target mask**.
The model no longer predicts random patches — it predicts specifically the part region.

In [ ]:
class PartImageNetDataset(Dataset):
    """
    PyTorch Dataset for PartImageNet with COCO-style JSON annotations.

    Each item returned is a triplet:
        image  : Tensor [3, img_size, img_size]  — normalised RGB image
        prompt : str                              — e.g. "Quadruped head"
        bbox   : Tensor [4]  (float32)            — [x, y, w, h] in pixels
                 before any resize; rescaled below to match img_size.

    Args:
        img_dir   : Path to the image folder (each image filename must match
                    the `file_name` field in the COCO JSON).
        ann_file  : Path to the COCO-format annotation JSON.
        img_size  : Target image size (square). Default 224.
        transform : Optional torchvision transform pipeline. If None a
                    sensible default (resize → tensor → normalise) is used.
        part_filter : Optional list of category names to keep.
                      If None, all parts are used.
    """

    # ImageNet mean/std — same normalisation used by the I-JEPA pretrained weights
    IMAGENET_MEAN = (0.485, 0.456, 0.406)
    IMAGENET_STD  = (0.229, 0.224, 0.225)

    def __init__(
        self,
        img_dir: str,
        ann_file: str,
        img_size: int = 224,
        transform=None,
        part_filter=None,
    ):
        super().__init__()
        self.img_dir  = Path(img_dir)
        self.img_size = img_size

        # ── Load COCO JSON ───────────────────────────────────────────────────
        with open(ann_file, "r") as f:
            coco = json.load(f)

        # ── Build category_id → name lookup ─────────────────────────────────
        self.cat_id_to_name = {
            cat["id"]: cat["name"] for cat in coco["categories"]
        }

        # ── Build image_id → file_name lookup ────────────────────────────────
        self.img_id_to_info = {
            img["id"]: img for img in coco["images"]
        }

        # ── Filter annotations ───────────────────────────────────────────────
        # COCO bbox format: [x_topleft, y_topleft, width, height] in pixels.
        # We skip degenerate boxes (w or h == 0).
        allowed_cats = set()
        if part_filter is not None:
            allowed_cats = {
                cid for cid, name in self.cat_id_to_name.items()
                if name in part_filter
            }

        self.samples = []  # list of (image_info_dict, annotation_dict)
        for ann in coco["annotations"]:
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0:
                continue                         # skip degenerate boxes
            if part_filter and ann["category_id"] not in allowed_cats:
                continue                         # skip unwanted parts
            img_info = self.img_id_to_info.get(ann["image_id"])
            if img_info is None:
                continue
            self.samples.append((img_info, ann))

        print(f"[PartImageNet] Loaded {len(self.samples)} part-annotated samples.")

        # ── Transform pipeline ───────────────────────────────────────────────
        if transform is not None:
            self.transform = transform
        else:
            self.transform = T.Compose([
                T.Resize((img_size, img_size)),  # resize to square
                T.ToTensor(),                    # HWC uint8 → CHW float [0,1]
                T.Normalize(                     # ImageNet normalisation
                    mean=self.IMAGENET_MEAN,
                    std=self.IMAGENET_STD,
                ),
            ])

    # ── Helpers ──────────────────────────────────────────────────────────────

    def _rescale_bbox(
        self, bbox, orig_w: int, orig_h: int
    ) -> torch.Tensor:
        """
        Rescale COCO pixel bbox to the resized image coordinate system.

        COCO stores bbox in *original* image pixels.  After resizing to
        (img_size × img_size) we must rescale so that downstream code
        can map bbox → patch indices correctly.

        Returns: Tensor([x, y, w, h]) in rescaled pixel space.
        """
        x, y, w, h = bbox
        scale_x = self.img_size / orig_w
        scale_y = self.img_size / orig_h
        return torch.tensor(
            [x * scale_x, y * scale_y, w * scale_x, h * scale_y],
            dtype=torch.float32,
        )

    # ── __len__ ───────────────────────────────────────────────────────────────

    def __len__(self) -> int:
        return len(self.samples)

    # ── __getitem__ ───────────────────────────────────────────────────────────

    def __getitem__(self, idx: int):
        """
        Returns
        -------
        image  : Tensor [3, img_size, img_size]    — normalised RGB
        prompt : str                                — part name as text
        bbox   : Tensor [4]  float32               — [x, y, w, h] rescaled
        """
        img_info, ann = self.samples[idx]

        # ── 1. Load image ─────────────────────────────────────────────────────
        img_path = self.img_dir / img_info["file_name"]
        image = Image.open(img_path).convert("RGB")   # always 3-channel
        orig_w, orig_h = image.size                   # PIL: (W, H)

        # ── 2. Apply transforms ───────────────────────────────────────────────
        # After this, image: Tensor [3, img_size, img_size]
        image = self.transform(image)

        # ── 3. Build text prompt ──────────────────────────────────────────────
        # Format: "<supercategory> <part_name>" → e.g. "Quadruped head"
        # The category name in PartImageNet already includes this (e.g.
        # "Quadruped_head"); we clean it up for the text encoder.
        raw_name = self.cat_id_to_name[ann["category_id"]]
        prompt = raw_name.replace("_", " ")   # "Quadruped_head" → "Quadruped head"

        # ── 4. Rescale bounding box ───────────────────────────────────────────
        # bbox: Tensor [4] = [x, y, w, h] in img_size pixel space
        bbox = self._rescale_bbox(ann["bbox"], orig_w, orig_h)

        return image, prompt, bbox

### Custom Collate Function

The default `torch.utils.data.default_collate` cannot batch a list of strings.
We write a minimal collator that:
- stacks images into `[B, 3, H, W]`
- keeps prompts as a plain Python list (the tokenizer will handle them)
- stacks bboxes into `[B, 4]`

In [ ]:
# ── Quick Sanity Check & Kaggle Compatibility ─────────────────────────────────
# We use the turkeyju/PartImageNet dataset from HuggingFace.
# On Kaggle, we download/extract it to /kaggle/working if it doesn't exist.
# We then dynamically locate the extracted dataset directory.

import os
from pathlib import Path

WORK_DIR = Path("/kaggle/working") if os.path.exists("/kaggle/working") else Path(".")

# 1. Search for val.json to see if we already have it
val_jsons = list(WORK_DIR.rglob("val.json"))

# 2. Download and extract if missing
if not val_jsons:
    print(f"Dataset not found in {WORK_DIR}. Downloading from HuggingFace...")
    try:
        from huggingface_hub import hf_hub_download
        import zipfile
        
        # Download the zip file
        zip_path = hf_hub_download(repo_id="turkeyju/PartImageNet", repo_type="dataset", filename="PartImageNet_Seg.zip")
        
        # Extract the zip file
        print(f"Extracting {zip_path} to {WORK_DIR}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(WORK_DIR)
            
        print("Extraction complete!")
        # Re-search after extraction
        val_jsons = list(WORK_DIR.rglob("val.json"))
        
    except ImportError:
        print("Please install huggingface_hub: !pip install huggingface_hub")

if val_jsons:
    # val.json is in <ROOT>/annotations/val.json, so the ROOT is parent.parent
    PART_IMAGENET_ROOT = val_jsons[0].parent.parent
    print(f"Successfully resolved dataset structure at: {PART_IMAGENET_ROOT}")
else:
    raise FileNotFoundError("Could not find val.json in the extracted dataset. The zip might be structured differently.")

ANN_FILE           = PART_IMAGENET_ROOT / "annotations" / "val.json"
IMG_DIR            = PART_IMAGENET_ROOT / "images"
if (IMG_DIR / "val").exists():
    IMG_DIR = IMG_DIR / "val"
IMG_SIZE           = 224
PATCH_SIZE         = 16
GRID_SIZE          = IMG_SIZE // PATCH_SIZE        # = 14 (14×14 patch grid)
NUM_PATCHES        = GRID_SIZE * GRID_SIZE         # = 196

print(f"Patch grid : {GRID_SIZE}×{GRID_SIZE} = {NUM_PATCHES} patches")

# Initialise dataset — runs even if the path doesn't exist yet (for dry-run).
if ANN_FILE.exists():
    dataset = PartImageNetDataset(
        img_dir  = IMG_DIR,
        ann_file = ANN_FILE,
        img_size = IMG_SIZE,
    )
    loader = DataLoader(
        dataset,
        batch_size  = 4,
        shuffle     = True,
        num_workers = 2,
        collate_fn  = part_collate_fn,
        pin_memory  = (device.type == "cuda"),
    )
    # ── Inspect one batch ───────────────────────────────────────────────────
    images, prompts, bboxes = next(iter(loader))
    print(f"images  shape : {images.shape}")
    # Shape: [4, 3, 224, 224]  → batch of 4 RGB images
    print(f"bboxes  shape : {bboxes.shape}")
    # Shape: [4, 4]            → [x, y, w, h] per image
    print(f"prompts       : {prompts}")
    # e.g. ['Quadruped head', 'Bird wing', 'Fish body', 'Snake head']
else:
    print("[INFO] PartImageNet path not found — skipping live load.")
    print("       Update PART_IMAGENET_ROOT to run on real data.")
    # Synthetic dry-run tensors so all downstream cells still execute.
    B = 4
    images  = torch.randn(B, 3, IMG_SIZE, IMG_SIZE)
    prompts = ["Quadruped head", "Bird wing", "Fish body", "Snake head"]
    # Fake bbox: centre square covering roughly 1/4 of image
    bboxes  = torch.tensor([
        [40., 40., 144., 144.],
        [20., 60., 100.,  80.],
        [50., 30., 120., 100.],
        [60., 80.,  80.,  60.],
    ])

### Visualise a Sample (Optional)
Draw the bounding box over the raw image to confirm the loader is working.

In [ ]:
def denormalise(tensor):
    """Undo ImageNet normalisation for display."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)

fig, axes = plt.subplots(1, min(4, len(images)), figsize=(16, 4))
for i, ax in enumerate(axes):
    img_np = denormalise(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img_np)
    x, y, w, h = bboxes[i].tolist()
    rect = mpatches.Rectangle(
        (x, y), w, h,
        linewidth=2, edgecolor="red", facecolor="none",
    )
    ax.add_patch(rect)
    ax.set_title(prompts[i], fontsize=10)
    ax.axis("off")
plt.suptitle("PartImageNet samples — red box = target part region", fontsize=12)
plt.tight_layout()
plt.show()

---
## Task 2 — The Text-Conditioned Context Encoder

Here we implement the **TI-JEPA style visual-text fusion**. The standard I-JEPA Context Encoder processes the unmasked image patches into a sequence of block representations.

We wrap this encoder inside our `TextConditionedContextEncoder`.
1. The visual context patches (Output of `VisionTransformer`) act as the **Queries (Q)**.
2. The frozen CLIP text embeddings act as the **Keys (K)** and **Values (V)**.
3. A **Cross-Attention** layer fuses the text semantic knowledge into the visual tokens.

We use `nn.MultiheadAttention` to perform this Text-to-Image (t2i) cross-attention.

In [ ]:
class TextConditionedContextEncoder(nn.Module):
    """
    Wraps the standard I-JEPA VisionTransformer with a text-conditioning pathway.
    
    Architecture:
      1) Image -> ViT Encoder -> Visual Context Tokens (Queries)
      2) Text Prompt -> CLIP Encoder -> Text Tokens (Keys/Values)
      3) Multihead Cross-Attention: Fuses text into visual context.
    """
    def __init__(
        self, 
        visual_encoder: VisionTransformer,  # The local I-JEPA vit object
        clip_model_name: str = "openai/clip-vit-base-patch32",
        num_heads: int = 8,
        dropout: float = 0.1,
    ):
        super().__init__()
        # ── 1. Visual Backbone ───────────────────────────────────────────────
        # We reuse the exact pre-trained/initialized local I-JEPA VisionTransformer.
        self.visual_encoder = visual_encoder
        embed_dim = self.visual_encoder.embed_dim   # e.g. 384 for vit_small
        
        # ── 2. Text Backbone (Frozen CLIP) ───────────────────────────────────
        self.tokenizer = CLIPTokenizer.from_pretrained(clip_model_name)
        self.text_encoder = CLIPTextModel.from_pretrained(clip_model_name)
        
        # FREEZE text encoder to prevent destroying its pre-trained semantics
        for param in self.text_encoder.parameters():
            param.requires_grad = False
            
        clip_hidden_size = self.text_encoder.config.hidden_size  # e.g., 512
        
        # ── 3. Projection layers to align dimensions ─────────────────────────
        # We project the text embeddings from CLIP dim (512) to ViT dim (384/768)
        self.text_proj = nn.Linear(clip_hidden_size, embed_dim)
        
        # ── 4. Cross-Attention (Text-to-Image) ───────────────────────────────
        # MultiheadAttention expects inputs of shape (Sequence, Batch, Embedding) 
        # by default, or (Batch, Sequence, Embedding) if batch_first=True.
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,  # Crucial for matching [B, N, D] shapes
        )
        
        # Layer Norms in the style of Transformer decoder blocks
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        # Simple Feed Forward Network (FFN) post-attention
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout)
        )

    def encode_text(self, prompts: list[str], device: torch.device):
        """
        Tokenize and encode the text prompts via CLIP.
        """
        # Tokenize (pads dynamically to max length in the batch)
        tokens = self.tokenizer(
            prompts, 
            padding=True, 
            return_tensors="pt"
        ).to(device)
        
        # Extract text sequence features
        # Shape: [batch_size, seq_len, clip_hidden_size] (e.g. [B, seq_len, 512])
        text_outputs = self.text_encoder(**tokens)
        text_features = text_outputs.last_hidden_state
        
        # Project to match ViT embed_dim
        # Shape: [batch_size, seq_len, embed_dim]
        text_features = self.text_proj(text_features)
        return text_features, tokens.attention_mask

    def forward(self, images, context_masks, prompts):
        """
        Forward pass of the Text-Conditioned Context Encoder.
        
        Args:
            images: FloatTensor [B, 3, H, W]
            context_masks: List of Tensors containing patch indices to keep.
                           Generated by MaskCollator.
            prompts: list of strings (length B) describing the body part.
            
        Returns:
            fused_context: [batch_size, num_context_patches, embed_dim]
            attn_weights: [batch_size, num_context_patches, text_seq_len] 
                          (useful for Visualization later)
        """
        batch_size = images.shape[0]
        device = images.device
        
        # ── 1. Visual Context Encoding ───────────────────────────────────────
        # We pass ONLY the context patches through the standard I-JEPA ViT.
        # x_v Shape: [B, num_context_patches, embed_dim]
        x_v = self.visual_encoder(images, context_masks)
        
        # ── 2. Text Feature Extraction ───────────────────────────────────────
        # x_t Shape: [B, text_seq_len, embed_dim]
        x_t, text_mask = self.encode_text(prompts, device)
        
        # MultiheadAttention's key_padding_mask expects True for positions to IGNORE.
        # text_mask is 1 for attend, 0 for pad. So we invert it.
        # key_padding_mask Shape: [B, text_seq_len] (Boolean)
        key_padding_mask = (text_mask == 0)
        
        # ── 3. Text-to-Image (T2I) Cross-Attention ───────────────────────────
        # Q = Visual Patches Context : [B, num_context_patches, embed_dim]
        # K = V = Text Features      : [B, text_seq_len, embed_dim]
        x_v_norm = self.norm1(x_v)
        
        # Output Shape: [B, num_context_patches, embed_dim]
        # attn_weights Shape: [B, num_context_patches, text_seq_len]
        attn_output, attn_weights = self.cross_attn(
            query=x_v_norm, 
            key=x_t, 
            value=x_t, 
            key_padding_mask=key_padding_mask
        )
        
        # Residual connection
        x_v_fused = x_v + attn_output
        
        # Apply FFN with residual
        x_v_fused = x_v_fused + self.ffn(self.norm2(x_v_fused))
        
        # x_v_fused Shape: [B, num_context_patches, embed_dim]
        return x_v_fused, attn_weights


In [ ]:
# ── Sanity Check: Instantiate the model & run forward pass ─────────────────

print("Initializing base I-JEPA VisionTransformer ('vit_small')...")
base_encoder = vit_small(
    img_size=[IMG_SIZE],
    patch_size=PATCH_SIZE,
).to(device)

print("Initializing Text-Conditioned Context Encoder...")
text_context_encoder = TextConditionedContextEncoder(
    visual_encoder=base_encoder,
    clip_model_name="openai/clip-vit-base-patch32",
).to(device)

# Generate dummy data if real isn't loaded
dummy_images = images.to(device)  # defined in Task 1

# Generate standard random context block masks using the local MaskCollator
collator = MaskCollator(
    input_size=(IMG_SIZE, IMG_SIZE),
    patch_size=PATCH_SIZE,
    pred_mask_scale=(0.2, 0.8), 
    enc_mask_scale=(0.85, 1.0),
    aspect_ratio=(0.3, 3.0),
    nenc=1,                   
    npred=4,
)

dummy_batch = [(torch.randn(3, IMG_SIZE, IMG_SIZE), torch.tensor([0])) for _ in range(images.size(0))]
_, dummy_masks_enc, _ = collator(dummy_batch)
dummy_masks_enc = [m.to(device) for m in dummy_masks_enc]

print(f"Generated context mask with {dummy_masks_enc[0].size(1)} patches.")

print("\nRunning Forward Pass...")
with torch.no_grad():
    fused_context, attn_weights = text_context_encoder(
        images=dummy_images,
        context_masks=dummy_masks_enc,
        prompts=prompts
    )

print(f"Output Fused Context Shape : {fused_context.shape}   ← [B, num_context_patches, embed_dim]")
print(f"Cross-Attention Map Shape  : {attn_weights.shape}    ← [B, num_context_patches, text_seq_len]")

---
## Task 3 — Semantic Target Masking & The Core Forward Pass

### 3A: Overriding Random Masking with Bounding-Box Masking

In the original I-JEPA the `MaskCollator` (in `src/masks/multiblock.py`) samples **random rectangular blocks** as prediction targets.  
We override this behaviour completely: instead of random blocks the **target mask** is the set of patch indices that lie inside the **PartImageNet bounding box**.

#### How we map a pixel bounding box to patch indices
```
Image: 224 x 224   Patch size: 16   Grid: 14 x 14 = 196 patches

bbox = [x, y, w, h]  (pixel coords, after rescaling to 224x224)

col_start = floor(x / patch_size)          col_end = ceil((x+w) / patch_size)
row_start = floor(y / patch_size)          row_end = ceil((y+h) / patch_size)

target_indices = { row * grid_w + col  |  row in [row_start, row_end),
                                          col in [col_start, col_end) }

context_indices = {0, 1, ..., 195} \ target_indices
```
This guarantees the model must **predict the semantic part** rather than an arbitrary rectangle.

### 3B: The Forward Pass
```
Context stream:  image + context_mask  -> TextConditionedContextEncoder -> fused_ctx
Target stream:   image (no mask)       -> Target Encoder (EMA, frozen)  -> h_target
Prediction:      fused_ctx + pos-encoded mask tokens -> Predictor      -> z_pred
Loss:            smooth_l1(z_pred, h_target[target_indices])
```

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3A — Semantic Target Masking:  bbox  →  patch index tensors
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def bbox_to_patch_masks(
    bboxes: torch.Tensor,
    img_size: int = 224,
    patch_size: int = 16,
    min_target_patches: int = 4,
):
    """
    Convert a batch of pixel-space bounding boxes into target/context patch
    index tensors that are directly compatible with I-JEPA's `apply_masks`.

    Args:
        bboxes : Tensor [B, 4]  — [x, y, w, h] in rescaled pixel coords
        img_size   : int  (224)
        patch_size : int  (16)
        min_target_patches : int — safety floor so target is never empty

    Returns:
        masks_target  : list of 1 Tensor [B, N_target] (patch indices inside bbox)
        masks_context : list of 1 Tensor [B, N_context] (patch indices outside bbox)
    """
    B = bboxes.size(0)
    grid_h = grid_w = img_size // patch_size  # 14
    total_patches = grid_h * grid_w            # 196

    target_indices_list = []
    context_indices_list = []

    for i in range(B):
        x, y, w, h = bboxes[i].tolist()

        # ── Map pixel bbox → patch grid coordinates ─────────────────────────
        # floor for start, ceil for end to cover every touched patch
        col_start = int(math.floor(x / patch_size))
        row_start = int(math.floor(y / patch_size))
        col_end   = int(math.ceil((x + w) / patch_size))
        row_end   = int(math.ceil((y + h) / patch_size))

        # Clamp to valid grid range [0, grid_size)
        col_start = max(0, min(col_start, grid_w - 1))
        row_start = max(0, min(row_start, grid_h - 1))
        col_end   = max(1, min(col_end,   grid_w))
        row_end   = max(1, min(row_end,   grid_h))

        # ── Build flat patch index list for the bbox region ──────────────────
        # Each patch index = row * grid_w + col  (row-major, matching ViT order)
        target_idx = []
        for r in range(row_start, row_end):
            for c in range(col_start, col_end):
                target_idx.append(r * grid_w + c)

        # Safety: ensure we always have at least min_target_patches
        if len(target_idx) < min_target_patches:
            # Expand bbox by 1 patch in each direction and retry
            row_start = max(0, row_start - 1)
            col_start = max(0, col_start - 1)
            row_end   = min(grid_h, row_end + 1)
            col_end   = min(grid_w, col_end + 1)
            target_idx = [r * grid_w + c
                          for r in range(row_start, row_end)
                          for c in range(col_start, col_end)]

        target_set = set(target_idx)
        # Context = everything NOT in the target
        context_idx = [p for p in range(total_patches) if p not in target_set]

        target_indices_list.append(torch.tensor(sorted(target_idx), dtype=torch.long))
        context_indices_list.append(torch.tensor(context_idx, dtype=torch.long))

    # ── Pad to uniform length within the batch ──────────────────────────────
    # apply_masks expects [B, N] tensors, so we pad shorter lists with the
    # last valid index (safe because gather just duplicates a token).
    max_target  = max(len(t) for t in target_indices_list)
    max_context = max(len(c) for c in context_indices_list)

    def pad_to(tensor_list, length):
        padded = []
        for t in tensor_list:
            if len(t) < length:
                pad = t[-1:].expand(length - len(t))
                t = torch.cat([t, pad])
            padded.append(t)
        return torch.stack(padded)  # [B, length]

    masks_target  = [pad_to(target_indices_list,  max_target)]
    masks_context = [pad_to(context_indices_list, max_context)]

    return masks_target, masks_context


# ── Quick test ──────────────────────────────────────────────────────────────
test_bboxes = torch.tensor([
    [40., 40., 144., 144.],   # roughly centre
    [0.,  0.,  64.,  64.],    # top-left corner
])
m_tgt, m_ctx = bbox_to_patch_masks(test_bboxes, img_size=224, patch_size=16)
print(f"Target mask shape : {m_tgt[0].shape}   ← [B, N_target]")
print(f"Context mask shape: {m_ctx[0].shape}  ← [B, N_context]")
print(f"Target indices (sample 0): {m_tgt[0][0].tolist()[:12]}...")
print(f"Sum check: {m_tgt[0].shape[1]} target + {m_ctx[0].shape[1]} context"
      f" ≈ {m_tgt[0].shape[1] + m_ctx[0].shape[1]} (total patches = {(224//16)**2})")

### Visualise the Semantic Mask
Red = target patches (the part the model must predict). Blue = context patches.

In [ ]:
def visualise_patch_masks(bboxes, img_size=224, patch_size=16):
    masks_t, masks_c = bbox_to_patch_masks(bboxes, img_size, patch_size)
    grid = img_size // patch_size
    B = bboxes.size(0)
    fig, axes = plt.subplots(1, B, figsize=(4*B, 4))
    if B == 1: axes = [axes]
    for i, ax in enumerate(axes):
        canvas = np.zeros((grid, grid, 3))
        for idx in masks_c[0][i].tolist():
            r, c = divmod(idx, grid)
            canvas[r, c] = [0.2, 0.4, 0.8]   # blue = context
        for idx in masks_t[0][i].tolist():
            r, c = divmod(idx, grid)
            canvas[r, c] = [0.9, 0.2, 0.2]   # red = target
        ax.imshow(canvas, interpolation='nearest')
        ax.set_title(f'Sample {i}', fontsize=10)
        ax.axis('off')
    plt.suptitle('Semantic Mask: RED=target (part), BLUE=context', fontsize=11)
    plt.tight_layout()
    plt.show()

visualise_patch_masks(bboxes[:2])

---
### 3B — The Core Forward Pass

This cell contains the **complete forward pass** that ties all modules together:

1. **`bbox_to_patch_masks`** converts bboxes → `masks_target` / `masks_context`
2. **Context Encoder** (text-conditioned): encodes visible patches + fuses text
3. **Predictor**: takes fused context + positional mask tokens → predicts target representations
4. **Target Encoder** (EMA, frozen): encodes full image, then gathers target patch features
5. **Loss**: Smooth-L1 between predicted and target representations

#### Critical Detail — Positional Embedding Extraction
The I-JEPA Predictor (`VisionTransformerPredictor`) has its own `predictor_pos_embed` of shape `[1, 196, pred_embed_dim]` — one embedding per grid position.

When we create the mask tokens for the **target region**, we must extract **exactly the positional embeddings** that correspond to the bounding-box patch grid locations so the predictor knows **where** in the image it is predicting:
```python
# masks_target[0] contains indices like [44, 45, 46, 58, 59, 60, ...]
# These are the row-major patch positions within the 14x14 grid
# We use apply_masks to gather the corresponding positional embeddings
pos_embs = apply_masks(predictor.predictor_pos_embed.repeat(B,1,1), masks_target)
```
These gathered positional embeddings are added to the **learnable mask token** (`predictor.mask_token`) before being concatenated with the context stream and fed through the predictor transformer blocks.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3B — PartConditionedIJEPA:  Full model forward pass
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class PartConditionedIJEPA(nn.Module):
    """
    Full Part-Conditioned I-JEPA model.

    Components (all imported from local I-JEPA repo):
      - context_encoder : TextConditionedContextEncoder  (from Task 2)
      - predictor       : VisionTransformerPredictor     (from src/models/)
      - target_encoder  : VisionTransformer              (EMA copy, frozen)

    The forward pass:
      1. Convert bboxes → semantic patch masks
      2. Encode context patches with text fusion
      3. Project fused context into predictor space
      4. Build mask tokens with target positional embeddings
      5. Run predictor to predict target representations
      6. Get ground-truth from frozen target encoder
      7. Compute loss
    """

    def __init__(
        self,
        context_encoder: TextConditionedContextEncoder,
        predictor: VisionTransformerPredictor,
        target_encoder: VisionTransformer,
        img_size: int = 224,
        patch_size: int = 16,
    ):
        super().__init__()
        self.context_encoder = context_encoder
        self.predictor = predictor
        self.target_encoder = target_encoder
        self.img_size = img_size
        self.patch_size = patch_size

        # Freeze the target encoder (updated via EMA externally)
        for p in self.target_encoder.parameters():
            p.requires_grad = False

    def forward(self, images, prompts, bboxes):
        """
        Args:
            images  : [B, 3, H, W]
            prompts : list[str] length B
            bboxes  : [B, 4]  — [x, y, w, h] in rescaled pixel coords

        Returns:
            loss         : scalar
            attn_weights : [B, N_ctx, text_seq_len]  (for visualisation)
        """
        B = images.size(0)
        device = images.device

        # ════════════════════════════════════════════════════════════════════
        # Step 1: Convert bounding boxes → patch-level masks
        # ════════════════════════════════════════════════════════════════════
        # masks_target  : list of 1 Tensor [B, N_target]  — indices of part patches
        # masks_context : list of 1 Tensor [B, N_context] — indices of remaining patches
        masks_target, masks_context = bbox_to_patch_masks(
            bboxes, self.img_size, self.patch_size
        )
        masks_target  = [m.to(device) for m in masks_target]
        masks_context = [m.to(device) for m in masks_context]

        # ════════════════════════════════════════════════════════════════════
        # Step 2: Context Encoder (Text-Conditioned)
        # ════════════════════════════════════════════════════════════════════
        # Feed ONLY the context (non-target) patches through the ViT + t2i fusion.
        # fused_ctx Shape: [B, N_context, embed_dim]
        # attn_weights Shape: [B, N_context, text_seq_len]
        fused_ctx, attn_weights = self.context_encoder(
            images, masks_context, prompts
        )

        # ════════════════════════════════════════════════════════════════════
        # Step 3: Predictor — predict target representations
        # ════════════════════════════════════════════════════════════════════
        # The VisionTransformerPredictor.forward(x, masks_x, masks) does:
        #   a) Projects x from encoder dim → predictor dim
        #   b) Adds positional embeddings for context positions (masks_x)
        #   c) Creates mask tokens, adds positional embeddings for TARGET
        #      positions (masks)
        #   d) Concatenates context + mask tokens, runs transformer blocks
        #   e) Returns only the mask-token outputs, projected back to encoder dim
        #
        # CRITICAL: Here masks_x=masks_context tells the predictor WHERE
        #           the context tokens sit in the grid (for pos-embed lookup).
        #           masks=masks_target tells it WHERE the target tokens sit
        #           (for pos-embed lookup on the mask tokens).
        #
        # Inside the predictor this is what happens with positional embeddings:
        #   predictor_pos_embed : [1, 196, pred_dim]  (full grid)
        #
        #   # For context tokens:
        #   x_pos = apply_masks(pos_embed.repeat(B,1,1), masks_context)
        #   #   → gathers pos embeds at context grid locations
        #   #   Shape: [B, N_context, pred_dim]
        #   x += x_pos
        #
        #   # For target (mask) tokens:
        #   pos_embs = apply_masks(pos_embed.repeat(B,1,1), masks_target)
        #   #   → gathers pos embeds at BBOX grid locations
        #   #   Shape: [B, N_target, pred_dim]
        #   pred_tokens = mask_token.repeat(B, N_target, 1)  # learnable
        #   pred_tokens += pos_embs   # <-- THIS encodes WHERE the part is
        #
        # z_pred Shape: [B, N_target, embed_dim]
        z_pred = self.predictor(
            fused_ctx,      # [B, N_context, embed_dim]
            masks_context,  # tells predictor where context sits in grid
            masks_target,   # tells predictor where target sits in grid
        )

        # ════════════════════════════════════════════════════════════════════
        # Step 4: Target Encoder (EMA, frozen) — ground truth
        # ════════════════════════════════════════════════════════════════════
        with torch.no_grad():
            # Encode the FULL image (no masking) through the target encoder
            # h_full Shape: [B, 196, embed_dim]
            h_full = self.target_encoder(images)

            # Normalise over feature dimension (standard I-JEPA practice)
            h_full = F.layer_norm(h_full, (h_full.size(-1),))

            # Gather ONLY the target-region patch features
            # h_target Shape: [B, N_target, embed_dim]
            h_target = apply_masks(h_full, masks_target)

        # ════════════════════════════════════════════════════════════════════
        # Step 5: Loss — Smooth L1 between prediction and target
        # ════════════════════════════════════════════════════════════════════
        # z_pred   Shape: [B, N_target, embed_dim]
        # h_target Shape: [B, N_target, embed_dim]
        loss = F.smooth_l1_loss(z_pred, h_target)

        return loss, attn_weights


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Sanity Check — full forward pass
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

EMBED_DIM       = 384   # vit_small
PRED_EMBED_DIM  = 192
PRED_DEPTH      = 6

# 1. Context encoder (reuse from Task 2, or build fresh)
base_enc = vit_small(img_size=[IMG_SIZE], patch_size=PATCH_SIZE).to(device)
ctx_encoder = TextConditionedContextEncoder(
    visual_encoder=base_enc,
    clip_model_name="openai/clip-vit-base-patch32",
).to(device)

# 2. Predictor (imported directly from I-JEPA)
predictor = vit_predictor(
    num_patches=NUM_PATCHES,     # 196
    embed_dim=EMBED_DIM,         # must match encoder output
    predictor_embed_dim=PRED_EMBED_DIM,
    depth=PRED_DEPTH,
    num_heads=6,
).to(device)

# 3. Target encoder — deep copy of the base encoder (will be EMA-updated)
target_enc = copy.deepcopy(base_enc).to(device)

# 4. Assemble
model = PartConditionedIJEPA(
    context_encoder=ctx_encoder,
    predictor=predictor,
    target_encoder=target_enc,
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
).to(device)

# Move data to device
test_images  = images.to(device)
test_bboxes  = bboxes.to(device)
test_prompts = prompts

print("Running full forward pass...")
loss, attn_w = model(test_images, test_prompts, test_bboxes)

print(f"\n✓ Loss          : {loss.item():.4f}")
print(f"✓ Attn weights  : {attn_w.shape}  ← [B, N_context, text_seq_len]")
print(f"\nTrainable params:")
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total     : {total:,}")
print(f"  Trainable : {trainable:,}")
print(f"  Frozen    : {total - trainable:,}  (target encoder + CLIP text)")

---
## Task 4 — Visualization & Sanity Check: Zero-Shot Part Localization

### What This Proves
The **cross-attention weights** from the t2i layer (Task 2) tell us exactly
how much each **visual patch** attended to the **text prompt**.

If our architecture works correctly, the attention should activate
**highest around the specific body part** mentioned in the prompt
(e.g. "Quadruped head" should light up the head region),
even without any explicit localization supervision — this is
**zero-shot part localization**.

#### How the attention map is constructed
```
attn_weights: [B, N_context_patches, text_seq_len]

Step 1: Average across text tokens → [B, N_context_patches]
        (each patch gets one scalar = "how much did it attend to the text?")

Step 2: Since context patches are a SUBSET of the 14×14 grid,
        we scatter the values back into a full 14×14 map.

Step 3: Upsample 14×14 → 224×224 via bicubic interpolation.

Step 4: Overlay as a heatmap on the original image.
```
Regions with **high attention** (warm colours) correspond to patches
the model found most relevant to the text — i.e. the body part.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Task 4 — Cross-Attention Visualization (Zero-Shot Part Localization)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_and_plot_attention(
    model: PartConditionedIJEPA,
    images: torch.Tensor,      # [B, 3, H, W]
    prompts: list,             # list[str] length B
    bboxes: torch.Tensor,      # [B, 4]
    img_size: int = 224,
    patch_size: int = 16,
    samples_to_show: int = 4,
):
    """
    Run a forward pass, extract cross-attention weights from the t2i layer,
    and overlay them as heatmaps on the original images.

    This serves as a ZERO-SHOT PART LOCALIZATION check:
    if the text prompt says 'Quadruped head', the heatmap should
    highlight the head region of the animal.
    """
    grid_size = img_size // patch_size  # 14
    device = images.device
    B = min(images.size(0), samples_to_show)

    # ── 1. Forward pass to collect attention weights ─────────────────────
    model.eval()
    with torch.no_grad():
        loss, attn_weights = model(images[:B], prompts[:B], bboxes[:B])
    # attn_weights Shape: [B, N_context_patches, text_seq_len]

    # ── 2. Get context mask indices (needed to scatter back to full grid) ─
    _, masks_context = bbox_to_patch_masks(
        bboxes[:B], img_size, patch_size
    )
    # masks_context[0] Shape: [B, N_context]
    context_indices = masks_context[0]  # [B, N_context]

    # ── 3. Average attention across text tokens ──────────────────────────
    # From [B, N_context, text_seq_len] → [B, N_context]
    attn_per_patch = attn_weights.mean(dim=-1).cpu()  # [B, N_context]

    # ── 4. Scatter into full 14×14 grid ─────────────────────────────────
    attn_maps = torch.zeros(B, grid_size * grid_size)  # [B, 196]
    for i in range(B):
        idx = context_indices[i].cpu().long()
        # Only fill positions that are valid (not padding)
        n_valid = min(len(idx), attn_per_patch.size(1))
        attn_maps[i, idx[:n_valid]] = attn_per_patch[i, :n_valid]

    # Reshape to 2D grid: [B, 14, 14]
    attn_maps = attn_maps.view(B, grid_size, grid_size)

    # ── 5. Upsample to image resolution ─────────────────────────────────
    # [B, 1, 14, 14] → [B, 1, 224, 224]
    attn_upsampled = F.interpolate(
        attn_maps.unsqueeze(1),
        size=(img_size, img_size),
        mode='bicubic',
        align_corners=False,
    ).squeeze(1)  # [B, 224, 224]

    # Normalise each map to [0, 1] for display
    for i in range(B):
        a = attn_upsampled[i]
        attn_upsampled[i] = (a - a.min()) / (a.max() - a.min() + 1e-8)

    # ── 6. Plot ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(3, B, figsize=(5 * B, 14))
    if B == 1:
        axes = axes.reshape(-1, 1)

    for i in range(B):
        # Row 0: Original image with bounding box
        img_np = denormalise(images[i].cpu()).permute(1, 2, 0).numpy()
        axes[0, i].imshow(img_np)
        x, y, w, h = bboxes[i].cpu().tolist()
        rect = mpatches.Rectangle(
            (x, y), w, h,
            linewidth=2, edgecolor='lime', facecolor='none', linestyle='--',
        )
        axes[0, i].add_patch(rect)
        axes[0, i].set_title(f'Input + GT bbox\n"{prompts[i]}"', fontsize=10)
        axes[0, i].axis('off')

        # Row 1: Raw attention heatmap (14×14 upsampled)
        axes[1, i].imshow(attn_upsampled[i].numpy(), cmap='hot', interpolation='bicubic')
        axes[1, i].set_title('Cross-Attention Heatmap', fontsize=10)
        axes[1, i].axis('off')

        # Row 2: Overlay — attention heatmap on original image
        axes[2, i].imshow(img_np)
        axes[2, i].imshow(
            attn_upsampled[i].numpy(),
            cmap='jet', alpha=0.5, interpolation='bicubic',
        )
        # Draw bbox on overlay too
        rect2 = mpatches.Rectangle(
            (x, y), w, h,
            linewidth=2, edgecolor='lime', facecolor='none', linestyle='--',
        )
        axes[2, i].add_patch(rect2)
        axes[2, i].set_title('Overlay: Attention + Image', fontsize=10)
        axes[2, i].axis('off')

    plt.suptitle(
        'Zero-Shot Part Localization via T2I Cross-Attention\n'
        '(Warm regions = high text-visual alignment → predicted part location)',
        fontsize=13, fontweight='bold',
    )
    plt.tight_layout()
    plt.show()

    return attn_maps  # [B, 14, 14] for further analysis


In [ ]:
# ── Run Zero-Shot Part Localization Visualization ────────────────────────────

print("Extracting cross-attention maps and visualizing...\n")
attn_maps = extract_and_plot_attention(
    model=model,
    images=test_images,
    prompts=test_prompts,
    bboxes=test_bboxes,
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    samples_to_show=4,
)

print(f"\nReturned attention maps shape: {attn_maps.shape}  ← [B, 14, 14]")
print("\n" + "="*70)
print("INTERPRETATION GUIDE")
print("="*70)
print("• Row 1: Original image with the ground-truth bounding box (green dashed).")
print("• Row 2: Raw cross-attention heatmap — brighter = higher text-visual alignment.")
print("• Row 3: Overlay of the heatmap on the image.")
print("")
print("If the model is working correctly, the HOT regions in the heatmap")
print("should roughly align with the GREEN bounding box, proving that the")
print("text prompt (e.g. 'Quadruped head') successfully guided the model's")
print("visual attention to that specific body part.")
print("")
print("NOTE: With random weights (no training), the attention will be diffuse.")
print("After training, you should see sharp, localized activation on the part.")

---
## Summary

| Component | Source | Role |
|-----------|--------|------|
| `VisionTransformer` | Local I-JEPA `src/models/vision_transformer.py` | Context & Target encoder backbone |
| `VisionTransformerPredictor` | Local I-JEPA `src/models/vision_transformer.py` | Predicts target from context + mask tokens |
| `apply_masks` | Local I-JEPA `src/masks/utils.py` | Gathers patch tokens by index |
| `MaskCollator` | Local I-JEPA `src/masks/multiblock.py` | Original random masking (overridden in Task 3) |
| `CLIPTextModel` | HuggingFace `transformers` | Frozen text encoder for part prompts |
| `TextConditionedContextEncoder` | **Custom (Task 2)** | Wraps ViT + CLIP with t2i cross-attention |
| `bbox_to_patch_masks` | **Custom (Task 3)** | Converts pixel bboxes → semantic patch masks |
| `PartConditionedIJEPA` | **Custom (Task 3)** | Full model with forward pass & loss |
| `extract_and_plot_attention` | **Custom (Task 4)** | Zero-shot part localization via attention maps |

### Next Steps
- **Train** the model on full PartImageNet with EMA momentum scheduling
- **Evaluate** part localization accuracy by thresholding attention maps
- **Extend** to multi-part prediction (multiple bboxes per image)